# Análise Exploratória: Gastos dos Deputados Federais (Cota Parlamentar - CEAP)

**Contexto:** 2026 é ano de eleições gerais no Brasil (Presidente, Governadores, Senadores, Deputados Federais e Estaduais). Antes de olhar para os candidatos de 2026 (cujo registro só ocorre em agosto), faz sentido entender **como os atuais parlamentares usaram o dinheiro público durante o mandato** — a 57ª Legislatura (2023-2026).

**Fonte dos dados:** Portal de Dados Abertos da Câmara dos Deputados — Cota para o Exercício da Atividade Parlamentar (CEAP), que cobre despesas como passagens aéreas, combustível, manutenção de escritório, alimentação, divulgação parlamentar, entre outras.

> **Nota sobre deputados estaduais:** os gastos de deputados *estaduais* não têm uma base nacional unificada — cada Assembleia Legislativa estadual publica (ou não) seus próprios dados, em formatos diferentes. Por isso este projeto foca nos **deputados federais**, que têm uma base única, confiável e bem documentada. Dá pra expandir depois comparando com 1-2 assembleias estaduais específicas, se quiser.

**Perguntas que vamos responder:**
1. Qual foi o gasto total por ano na legislatura atual?
2. Quais categorias de despesa mais pesam no orçamento?
3. Como o gasto se distribui entre partidos e estados?
4. Quem são os deputados com maior e menor gasto?
5. Existe sazonalidade ou outliers relevantes?


## 1. Setup e Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import io

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 5)


## 2. Download dos dados

Os dados são publicados por ano, em arquivos CSV compactados, no padrão:
`https://www.camara.leg.br/cotas/Ano-{ano}.csv.zip`

A legislatura atual (57ª) vai de 2023 a 2026, então baixamos esses 4 anos.

⚠️ **Importante:** esta célula precisa de acesso à internet ao site da Câmara. Se você rodar em um ambiente sem acesso (como um sandbox restrito), baixe manualmente os arquivos em
https://www2.camara.leg.br/transparencia/cota-para-exercicio-da-atividade-parlamentar/dados-abertos-cota-parlamentar
e ajuste a função `carregar_ano` para ler de um arquivo local.

In [2]:
ANOS = [2023, 2024, 2025, 2026]

def carregar_ano(ano: int) -> pd.DataFrame:
    """Baixa e carrega o CSV de gastos da cota parlamentar de um ano específico."""
    url = f"https://www.camara.leg.br/cotas/Ano-{ano}.csv.zip"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        nome_arquivo = z.namelist()[0]
        with z.open(nome_arquivo) as f:
            df = pd.read_csv(f, sep=";", encoding="latin-1", low_memory=False)
    df["anoReferencia"] = ano
    return df

dfs = []
for ano in ANOS:
    try:
        dfs.append(carregar_ano(ano))
        print(f"✅ {ano}: carregado")
    except Exception as e:
        print(f"⚠️ {ano}: erro ao baixar ({e})")

gastos = pd.concat(dfs, ignore_index=True)
print(f"\nTotal de registros: {len(gastos):,}")


✅ 2023: carregado
✅ 2024: carregado
✅ 2025: carregado
✅ 2026: carregado

Total de registros: 748,828


## 3. Limpeza e seleção de colunas

In [3]:
# Colunas mais relevantes para a análise
colunas_uteis = [
    "txNomeParlamentar", "sgPartido", "sgUF", "nuLegislatura",
    "txtDescricao", "txtFornecedor", "txtCNPJCPF",
    "datEmissao", "numMes", "numAno", "vlrDocumento", "vlrGlosa", "vlrLiquido",
    "anoReferencia"
]
colunas_existentes = [c for c in colunas_uteis if c in gastos.columns]
gastos = gastos[colunas_existentes].copy()

# Tipos
gastos["vlrLiquido"] = pd.to_numeric(gastos["vlrLiquido"], errors="coerce")
gastos["datEmissao"] = pd.to_datetime(gastos["datEmissao"], errors="coerce")

# Remove despesas com valor líquido nulo, negativo (bilhetes de compensação) ou ausente
gastos = gastos[gastos["vlrLiquido"].notna() & (gastos["vlrLiquido"] > 0)]

print(gastos.shape)
gastos.head()


(721023, 13)


,sgPartido,sgUF,nuLegislatura,txtDescricao,txtFornecedor,txtCNPJCPF,datEmissao,numMes,numAno,vlrDocumento,vlrGlosa,vlrLiquido,anoReferencia
0,NaN,NaN,2023,MANUTENÃÃO DE ESCRITÃRIO DE APOIO Ã ATIVID...,AMORETTO CAFES EXPRESSO LTDA,085.324.290/0013-1,2023-02-07,2,2023,899.0,0.0,899.0,2023
1,NaN,NaN,2023,MANUTENÃÃO DE ESCRITÃRIO DE APOIO Ã ATIVID...,AMORETTO CAFES EXPRESSO LTDA,085.324.290/0013-1,2023-06-06,6,2023,899.0,0.0,899.0,2023
2,NaN,NaN,2023,MANUTENÃÃO DE ESCRITÃRIO DE APOIO Ã ATIVID...,AMORETTO CAFES EXPRESSO LTDA,085.324.290/0013-1,2023-07-12,7,2023,899.0,0.0,899.0,2023
3,NaN,NaN,2023,MANUTENÃÃO DE ESCRITÃRIO DE APOIO Ã ATIVID...,AMORETTO CAFES EXPRESSO LTDA,085.324.290/0013-1,2023-08-03,8,2023,899.0,0.0,899.0,2023
4,NaN,NaN,2023,MANUTENÃÃO DE ESCRITÃRIO DE APOIO Ã ATIVID...,AMORETTO CAFES EXPRESSO LTDA,085.324.290/0013-1,2023-09-12,9,2023,899.0,0.0,899.0,2023


In [ ]:
gastos.info()


## 4. Visão geral

Gasto total, número de deputados únicos e estatísticas descritivas do valor das despesas.

In [4]:
print(f"Gasto total no período: R$ {gastos['vlrLiquido'].sum():,.2f}")
print(f"Número de deputados únicos: {gastos['txNomeParlamentar'].nunique()}")
print(f"Número de notas/documentos: {len(gastos):,}")

gastos["vlrLiquido"].describe()


Gasto total no período: R$ 874,300,511.28


KeyError: 'txNomeParlamentar'

## 5. Gasto total por ano

In [ ]:
gasto_por_ano = gastos.groupby("anoReferencia")["vlrLiquido"].sum() / 1_000_000

ax = gasto_por_ano.plot(kind="bar", color="#4285F4")
ax.set_title("Gasto total com cota parlamentar por ano (R$ milhões)")
ax.set_xlabel("Ano")
ax.set_ylabel("R$ milhões")
plt.tight_layout()
plt.show()


## 6. Top categorias de despesa

In [ ]:
top_categorias = (
    gastos.groupby("txtDescricao")["vlrLiquido"]
    .sum()
    .sort_values(ascending=False)
    .head(10) / 1_000_000
)

ax = top_categorias.sort_values().plot(kind="barh", color="#34A853")
ax.set_title("Top 10 categorias de despesa (R$ milhões, 2023-2026)")
ax.set_xlabel("R$ milhões")
plt.tight_layout()
plt.show()


## 7. Gasto por partido

In [ ]:
gasto_partido = (
    gastos.groupby("sgPartido")["vlrLiquido"]
    .sum()
    .sort_values(ascending=False)
    .head(15) / 1_000_000
)

ax = gasto_partido.sort_values().plot(kind="barh", color="#FBBC05")
ax.set_title("Gasto total por partido — top 15 (R$ milhões)")
ax.set_xlabel("R$ milhões")
plt.tight_layout()
plt.show()


## 8. Gasto por estado (UF)

Lembrando: o valor da cota muda por estado (estados mais distantes de Brasília recebem cota maior por causa do custo das passagens aéreas), então é esperado ver variação aqui.

In [ ]:
gasto_uf = (
    gastos.groupby("sgUF")["vlrLiquido"]
    .sum()
    .sort_values(ascending=False) / 1_000_000
)

ax = gasto_uf.plot(kind="bar", color="#EA4335", figsize=(12, 5))
ax.set_title("Gasto total por estado de origem do deputado (R$ milhões)")
ax.set_xlabel("UF")
ax.set_ylabel("R$ milhões")
plt.tight_layout()
plt.show()


## 9. Top 10 deputados por gasto total

In [ ]:
top_deputados = (
    gastos.groupby(["txNomeParlamentar", "sgPartido", "sgUF"])["vlrLiquido"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
top_deputados


## 10. Evolução mensal e outliers

In [ ]:
gastos["anoMes"] = gastos["datEmissao"].dt.to_period("M")
evolucao_mensal = gastos.groupby("anoMes")["vlrLiquido"].sum() / 1_000_000

ax = evolucao_mensal.plot(kind="line", marker="o", color="#673AB7")
ax.set_title("Evolução mensal do gasto total (R$ milhões)")
ax.set_xlabel("Mês")
ax.set_ylabel("R$ milhões")
plt.tight_layout()
plt.show()


In [ ]:
# Outliers: distribuição do valor das despesas individuais (atenção à escala log)
plt.figure(figsize=(8, 4))
sns.boxplot(x=np.log10(gastos["vlrLiquido"]))
plt.title("Distribuição (log10) do valor das despesas individuais")
plt.xlabel("log10(valor da despesa em R$)")
plt.tight_layout()
plt.show()

# As 10 despesas individuais de maior valor
gastos.nlargest(10, "vlrLiquido")[
    ["txNomeParlamentar", "sgPartido", "sgUF", "txtDescricao", "txtFornecedor", "datEmissao", "vlrLiquido"]
]


## 11. Conclusões

*(Preencha com os principais insights depois de rodar com os dados reais — por exemplo: categoria que mais consome a cota, estado/partido com maior gasto médio por deputado, sazonalidade perto de datas-chave como recesso ou eleições, e outliers que merecem investigação manual.)*

**Possíveis próximos passos:**
- Calcular gasto **médio por deputado** (não só total), já que o número de deputados por UF/partido varia
- Cruzar com dados de **votos recebidos em 2022** para ver se há relação entre gasto e desempenho eleitoral
- Adicionar dados de **proposições e presença em votações** (também disponíveis na API da Câmara) para uma visão de produtividade vs. gasto
- Comparar com 1-2 Assembleias Legislativas estaduais, se os dados estiverem disponíveis em formato aberto
